# 02. Keypairs & Signing

`arctrust.keypair` is the cryptographic floor of every Arc deployment. Every
signature in the system — agent-to-agent message authentication, audit-chain
links, signed manifests, pairing approvals — passes through these three
functions: `generate_keypair`, `sign`, `verify`. There is exactly one Ed25519
implementation in Arc, and it lives here.

This notebook teaches the primitive by running it. We generate real keypairs
(they are cheap), produce real signatures, tamper with them, and watch
verification fail. We then load operator and issuer pubkeys from a TOML trust
store the way the gateway and backend loader do at runtime.

**You will learn:**
- Why Arc standardised on Ed25519 (federal-grade, deterministic, fast).
- How to generate, serialise, and reconstruct keypairs from a seed.
- The exact byte sizes Arc enforces (32-byte keys, 64-byte signatures).
- How `verify()` collapses every failure mode to `False` — never raises.
- How `load_operator_pubkey` / `load_issuer_pubkey` resolve a DID to a pubkey.
- How signing pairs with `AgentIdentity` from notebook 01.
- A defensible key-rotation pattern that preserves the audit trail.


## 1. Setup

Standard Arc notebook setup — locate the repo root and add every package's `src/` (and `tests/`) to `sys.path`. Also load any `.env` so we pick up dev defaults. No API keys required for this notebook; everything we do is local cryptography against `tmp_path`-style scratch directories.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Confirm the import surface we will exercise. Every symbol below is part of
the public API exported from `arctrust/__init__.py` — nothing in this notebook
reaches into private modules.

In [ ]:
from arctrust import (
    KeyPair,
    generate_keypair,
    sign,
    verify,
    load_operator_pubkey,
    load_issuer_pubkey,
    invalidate_cache,
    TrustStoreError,
    AgentIdentity,
    generate_did,
)
from arctrust.keypair import KEY_SIZE, SIGNATURE_SIZE

print("arctrust public API loaded")
print(f"  KEY_SIZE       = {KEY_SIZE} bytes")
print(f"  SIGNATURE_SIZE = {SIGNATURE_SIZE} bytes")

## 2. Why Ed25519

Arc could have picked any digital-signature scheme. Ed25519 was chosen because
every property it offers maps directly onto an Arc requirement:

| Property | Why it matters for Arc |
|---|---|
| **Deterministic signatures** | Same key + same message → same signature. No PRNG to fail. Reproducible audit chains. |
| **Small keys (32 bytes)** | Fits in a TOML cell, an HTTP header, a DID hash. Cheap to ship. |
| **Fast verify (~50 us)** | The gateway can verify thousands of pairing signatures per second without breaking a sweat. |
| **Side-channel resistant** | Constant-time scalar multiplication; safe to use on shared hardware. |
| **NIST FIPS 186-5 approved** | Federal-mode deployments need an algorithm in the FIPS catalog; Ed25519 is. |
| **No parameter choices** | One curve (edwards25519). One hash (SHA-512). One signature size (64 bytes). Nothing to misconfigure. |

Operationally, the rule is simple: **never reach for `nacl` directly inside
Arc code**. Arc has exactly one Ed25519 implementation, and it lives in
`arctrust.keypair`. Every other module — identity, audit, policy, trust store
— calls these three functions.

In [ ]:
# Quick sanity benchmark — sign and verify the same message a few thousand
# times so we can show the orders of magnitude. Real production verification
# happens at the same rate; nothing in arctrust adds artificial latency.
import time

kp = generate_keypair()
msg = b"arc-bench"

N = 2000
t0 = time.perf_counter()
for _ in range(N):
    s = sign(msg, kp.private_key)
sign_us = (time.perf_counter() - t0) / N * 1e6

t0 = time.perf_counter()
for _ in range(N):
    verify(msg, s, kp.public_key)
verify_us = (time.perf_counter() - t0) / N * 1e6

print(f"sign:   {sign_us:7.1f} us / op")
print(f"verify: {verify_us:7.1f} us / op")
print(f"(N={N} iterations on this kernel)")

## 3. Generating keypairs

`generate_keypair()` pulls 32 bytes from a cryptographically secure RNG (the
OS `/dev/urandom` via libsodium) and returns a frozen `KeyPair` dataclass.

Two structural rules to internalise:

1. The dataclass is `frozen=True`. You cannot mutate `kp.private_key` after
   construction. If you need a different key, generate a new pair.
2. Both fields are exactly 32 bytes. Anything else is a bug in the caller —
   `sign()` will raise `ValueError` and `verify()` will collapse to `False`.

In [ ]:
kp = generate_keypair()

print(type(kp).__name__)
print(f"public_key  ({len(kp.public_key)} bytes): {kp.public_key.hex()[:32]}...")
print(f"private_key ({len(kp.private_key)} bytes): <redacted>")
print(f"frozen?      {kp.__class__.__dataclass_params__.frozen}")

Two fresh calls give two independent keypairs. Generation is non-deterministic
by design — entropy comes from the OS RNG.

In [ ]:
kp1 = generate_keypair()
kp2 = generate_keypair()

assert kp1.public_key != kp2.public_key
assert kp1.private_key != kp2.private_key
print("two independent keypairs generated; public keys differ as expected")

### Reconstructing from a seed

Sometimes you need to load a stored private key — for example, when an agent
is restored from disk after restart. `KeyPair.from_seed(seed)` rebuilds the
same keypair deterministically. This is the only supported way to instantiate
a `KeyPair` from raw bytes.

In [ ]:
stored_seed = kp1.private_key  # in real life, this comes from a vault / file
rebuilt = KeyPair.from_seed(stored_seed)

assert rebuilt.public_key == kp1.public_key
assert rebuilt.private_key == kp1.private_key
print("rebuilt keypair matches original on both fields")

Wrong seed length is a programming error, not a runtime condition — Arc
raises immediately rather than producing a silently bogus key. Catching the
exception here so the cell completes; the demo is showing that bad input is
rejected at the boundary.

In [ ]:
try:
    KeyPair.from_seed(b"too-short")
except ValueError as exc:
    print(f"expected failure: {exc}")

## 4. Signing & verifying

`sign(message, private_key)` returns a 64-byte Ed25519 signature.
`verify(message, signature, public_key)` returns `True` if the signature was
produced by the matching private key over that exact message, else `False`.

Two things that catch new users off-guard:

1. `verify()` **never raises**. Wrong key length, malformed signature,
   tampered message — all collapse to `False`. This is intentional: callers
   want a boolean check, not a `try/except` ladder.
2. Signatures are deterministic. Sign the same message twice with the same
   key and you get the same 64 bytes. This is what makes audit hash chains
   reproducible across nodes.

In [ ]:
kp = generate_keypair()
msg = b'tool_call: read_file("/etc/passwd")'

sig = sign(msg, kp.private_key)
print(f"signature length: {len(sig)} bytes (must be {SIGNATURE_SIZE})")
print(f"signature head:   {sig.hex()[:32]}...")
print(f"verify:           {verify(msg, sig, kp.public_key)}")

Determinism, demonstrated.

In [ ]:
sig_a = sign(b"msg", kp.private_key)
sig_b = sign(b"msg", kp.private_key)
assert sig_a == sig_b
print("two independent calls produced byte-identical signatures")

Empty messages are valid. Federal audit chains use this property to seal a
genesis record before any payload exists.

In [ ]:
empty_sig = sign(b"", kp.private_key)
assert len(empty_sig) == SIGNATURE_SIZE
assert verify(b"", empty_sig, kp.public_key)
print("empty-message round trip ok")

Wrong private-key length on the **signing** side is a hard error — `sign()`
is the one place where bad input raises rather than collapsing.

In [ ]:
try:
    sign(b"msg", b"\x00" * 16)  # 16 bytes is wrong; must be 32
except ValueError as exc:
    print(f"sign raised as expected: {exc}")

## 5. Tampered messages

The whole point of a signature is that flipping a single bit in the message
invalidates the signature. Below we run that exact experiment so the failure
mode is concrete rather than abstract.

All three failure modes — wrong message, tampered signature, wrong key —
return `False`. None of them raise. This shape is what keeps every call site
in Arc clean: `if verify(...): ...` and you're done.

In [ ]:
kp = generate_keypair()
msg = b"transfer 100 ARC to alice"
sig = sign(msg, kp.private_key)

# Mutate one byte of the message
tampered_msg = b"transfer 100 ARC to malloRy"
ok = verify(tampered_msg, sig, kp.public_key)
print(f"verify(tampered message): {ok}  (expected False)")

Same idea, but mutating the **signature** instead. Useful for demonstrating
that even a single corrupted byte in transit breaks the chain.

In [ ]:
tampered_sig = bytearray(sig)
tampered_sig[0] ^= 0x01  # flip the lowest bit of byte 0
ok = verify(msg, bytes(tampered_sig), kp.public_key)
print(f"verify(1-bit flipped sig): {ok}  (expected False)")

Wrong **public key** — same shape, same answer. We are showing that all
three failure modes collapse cleanly to `False`; nothing here ever throws.

In [ ]:
other = generate_keypair()
ok = verify(msg, sig, other.public_key)
print(f"verify(wrong public key): {ok}  (expected False)")

# And just to make the no-raise property concrete:
ok = verify(msg, b"short-sig", kp.public_key)
print(f"verify(8-byte malformed sig): {ok}  (no exception raised)")

ok = verify(msg, sig, b"short-pubkey")
print(f"verify(11-byte malformed pubkey): {ok}  (no exception raised)")

## 6. Loading operator / issuer pubkeys

Arc verifies two distinct classes of signatures at runtime:

1. **Operator signatures** (`arcgateway.pairing`) — a human operator approves
   a DM-to-channel pairing by signing the challenge. The gateway needs the
   operator's pubkey to verify.
2. **Issuer signatures** (`arcrun.backends.loader`) — a trust authority signs
   the `allowed_backends` manifest. The loader needs the issuer's pubkey to
   verify before trusting the backend list.

Both resolve a DID to a 32-byte Ed25519 pubkey via two TOML files in
`~/.arc/trust/`: `operators.toml` and `issuers.toml`. Both files **must**
have `0o600` permissions on disk — anything looser is a `TrustStoreError`.

We will mock the trust dir under `tmp_path` so this notebook can run
anywhere, just like the production tests do.

In [ ]:
import base64
import os
import tempfile
from pathlib import Path

scratch = Path(tempfile.mkdtemp(prefix="arctrust-nb-"))
trust_dir = scratch / "trust"
trust_dir.mkdir()

print(f"trust_dir = {trust_dir}")

Write a real `operators.toml` with a freshly generated pubkey, exactly as a
deployment script would. Then chmod it to `0o600` — without this step the
loader rejects the file.

In [ ]:
from nacl.signing import VerifyKey  # type: ignore[import-not-found]

op_kp = generate_keypair()
op_did = generate_did(
    VerifyKey(op_kp.public_key),
    org="default",
    agent_type="operator",
)

op_file = trust_dir / "operators.toml"
op_file.write_text(
    f'[operators."{op_did}"]\n'
    f'public_key = "{base64.b64encode(op_kp.public_key).decode()}"\n'
    f'added_at = "2026-05-07T00:00:00Z"\n'
    f'notes = "Notebook demo operator"\n',
    encoding="utf-8",
)
os.chmod(op_file, 0o600)

print(f"wrote {op_file}")
print(f"op DID: {op_did}")

`load_operator_pubkey` resolves the DID to the 32-byte pubkey. The result is
exactly the bytes that came out of `generate_keypair()` — no encoding drift.

In [ ]:
invalidate_cache()  # always start tests / demos with a clean cache
loaded = load_operator_pubkey(op_did, trust_dir=trust_dir)

assert loaded == op_kp.public_key
print(f"loaded pubkey matches generator output ({len(loaded)} bytes)")

End-to-end: sign with the operator's private key, verify with the pubkey we
just loaded from the trust store. This is the exact code path the gateway
runs when validating a pairing approval.

In [ ]:
challenge = b"pair:#some-channel:2026-05-07T12:00:00Z:nonce-abc"
approval_sig = sign(challenge, op_kp.private_key)

trusted_pubkey = load_operator_pubkey(op_did, trust_dir=trust_dir)
ok = verify(challenge, approval_sig, trusted_pubkey)
print(f"pairing approval verifies: {ok}")

### Issuer pubkeys

`load_issuer_pubkey` is structurally identical, just reads `issuers.toml`.
This is what the backend manifest loader uses.

In [ ]:
iss_kp = generate_keypair()
iss_did = "did:arc:default:trust-authority/" + op_did.split("/")[-1][:8]

iss_file = trust_dir / "issuers.toml"
iss_file.write_text(
    f'[issuers."{iss_did}"]\n'
    f'public_key = "{base64.b64encode(iss_kp.public_key).decode()}"\n'
    f'added_at = "2026-05-07T00:00:00Z"\n'
    f'role = "manifest-signer"\n',
    encoding="utf-8",
)
os.chmod(iss_file, 0o600)

invalidate_cache()
iss_pubkey = load_issuer_pubkey(iss_did, trust_dir=trust_dir)
assert iss_pubkey == iss_kp.public_key
print("issuer pubkey loaded; ready to verify a signed backend manifest")

### Error cases — missing file, unknown DID, insecure perms

Every failure path returns a `TrustStoreError` with a structured `code` so
that callers can emit a clean audit event without parsing strings.
We catch and print to keep the notebook flowing — production would let the
exception propagate to the policy layer.

In [ ]:
# 1. Unknown DID
invalidate_cache()
try:
    load_operator_pubkey("did:arc:default:operator/00000000", trust_dir=trust_dir)
except TrustStoreError as exc:
    print(f"  unknown DID -> code={exc.code}")

# 2. Missing file (point at a brand-new empty dir)
empty_dir = scratch / "empty"
empty_dir.mkdir()
try:
    load_operator_pubkey(op_did, trust_dir=empty_dir)
except TrustStoreError as exc:
    print(f"  missing file -> code={exc.code}")

# 3. Insecure perms (chmod 0o644 — group and world readable)
loose_dir = scratch / "loose"
loose_dir.mkdir()
loose_file = loose_dir / "operators.toml"
loose_file.write_text(op_file.read_text(encoding="utf-8"), encoding="utf-8")
os.chmod(loose_file, 0o644)
invalidate_cache()
try:
    load_operator_pubkey(op_did, trust_dir=loose_dir)
except TrustStoreError as exc:
    print(f"  insecure perms -> code={exc.code}")

Malformed key material is caught with the same shape — `TRUST_STORE_BAD_KEY`
is what you will see in audit logs when an admin pastes the wrong base64
blob into the trust file.

In [ ]:
bad_dir = scratch / "badkey"
bad_dir.mkdir()
bad_file = bad_dir / "operators.toml"
bad_file.write_text(
    '[operators."did:arc:default:operator/aabbccdd"]\npublic_key = "!!!not-base64!!!"\n',
    encoding="utf-8",
)
os.chmod(bad_file, 0o600)
invalidate_cache()

try:
    load_operator_pubkey("did:arc:default:operator/aabbccdd", trust_dir=bad_dir)
except TrustStoreError as exc:
    print(f"  malformed key -> code={exc.code}")

## 7. Identity-bound signing

Notebook 01 introduced `AgentIdentity` — a DID + Ed25519 keypair bundled
together. Under the hood it is exactly a `KeyPair` with a deterministic DID
derived from the public key. Most production code signs *through*
`AgentIdentity` rather than calling `sign()` directly, so that audit events
can attribute every signature to a specific agent.

We will (a) build an `AgentIdentity`, (b) sign a synthetic tool-call payload,
and (c) verify with the same identity's public key. This is the canonical
shape for any inter-agent message in Arc.

In [ ]:
agent = AgentIdentity.generate(org="default", agent_type="executor")

print(f"agent DID:   {agent.did}")
print(f"can sign:    {agent.can_sign}")
print(f"pubkey size: {len(agent.public_key)} bytes")

Sign a tool-call payload as the agent. In practice this is what the agent
framework does on every dispatched tool call so the audit chain has a
verifiable origin.

In [ ]:
payload = b'{"tool": "read_file", "args": {"path": "/tmp/x"}}'

agent_sig = agent.sign(payload)
ok = verify(payload, agent_sig, agent.public_key)
print(f"agent signature length: {len(agent_sig)}")
print(f"verify with agent pubkey: {ok}")

And of course a different agent's pubkey will fail verification — this is
the property that prevents one agent from impersonating another in the
audit chain.

In [ ]:
other_agent = AgentIdentity.generate(org="default", agent_type="executor")
ok = verify(payload, agent_sig, other_agent.public_key)
print(f"verify with wrong agent pubkey: {ok}  (expected False)")

## 8. Key rotation patterns

Keys do get rotated — compromise, scheduled hygiene, personnel turnover.
The rule Arc enforces is: **never delete a retired pubkey, never reuse a
DID with a different keypair**. The trust store is append-only over its
lifetime; rotation means adding a new DID, not silently swapping a key.

The reason: every audit event is signed by the keypair active *at the time
the event was emitted*. If a verifier replaying yesterday's logs sees a
pubkey that no longer exists in the trust store, the chain breaks. A clean
rotation looks like this:

1. Generate a new keypair on the operator's machine.
2. Compute the new DID (different hash suffix → different DID).
3. Add a new entry to `operators.toml`. Leave the old entry in place.
4. Stop signing new approvals with the old key.
5. After the audit retention window (e.g. 7 years for federal), retire the
   old entry to a separate `archive/operators-retired.toml` so historical
   verification still works but live lookups don't see it.

The demo below shows steps 1–4.

In [ ]:
# Step 1: generate a new keypair for the operator (rotation event)
op_kp_v2 = generate_keypair()
from nacl.signing import VerifyKey  # type: ignore[import-not-found]

op_did_v2 = generate_did(VerifyKey(op_kp_v2.public_key), org="default", agent_type="operator")

assert op_did_v2 != op_did, "rotated keys must produce a new DID"
print(f"old DID: {op_did}")
print(f"new DID: {op_did_v2}")

Step 2 — append the new DID to `operators.toml` while keeping the old entry
intact. Both keys remain verifiable for any historical signatures that were
ever produced under them.

In [ ]:
import textwrap

appended = (
    textwrap.dedent(f'''
    [operators."{op_did_v2}"]
    public_key = "{base64.b64encode(op_kp_v2.public_key).decode()}"
    added_at = "2026-05-07T00:00:00Z"
    notes = "Rotation v2 — replaces {op_did}"
''').strip()
    + "\n"
)

# Read existing content, then write back the union
existing = op_file.read_text(encoding="utf-8")
op_file.write_text(existing + "\n" + appended, encoding="utf-8")
os.chmod(op_file, 0o600)

invalidate_cache()  # critical — TTL cache would otherwise hide the new entry

# Both old and new pubkeys resolve
assert load_operator_pubkey(op_did, trust_dir=trust_dir) == op_kp.public_key
assert load_operator_pubkey(op_did_v2, trust_dir=trust_dir) == op_kp_v2.public_key
print("both DIDs resolve cleanly; audit chain integrity preserved")

Step 3 — past signatures still verify under the old pubkey, and new
signatures verify under the new pubkey. Note that `invalidate_cache()` is
*always* called after a write to the trust store; the 60-second TTL is
there for read amplification, not as a write barrier.

In [ ]:
old_signed = sign(b"historical pairing", op_kp.private_key)
new_signed = sign(b"fresh pairing", op_kp_v2.private_key)

assert verify(b"historical pairing", old_signed, load_operator_pubkey(op_did, trust_dir=trust_dir))
assert verify(b"fresh pairing", new_signed, load_operator_pubkey(op_did_v2, trust_dir=trust_dir))
print("historical and rotated signatures both verify against the trust store")

## 9. Summary

Three primitives, one trust store, one rule:

- `generate_keypair()` — fresh Ed25519 keypair from the OS RNG.
- `KeyPair.from_seed(seed)` — deterministic reconstruction from a 32-byte seed.
- `sign(message, private_key)` — 64-byte Ed25519 signature, deterministic.
- `verify(message, signature, public_key)` — boolean; never raises.
- `load_operator_pubkey(did, trust_dir=...)` — DID → 32-byte pubkey from `operators.toml`.
- `load_issuer_pubkey(did, trust_dir=...)` — same shape, reads `issuers.toml`.
- `invalidate_cache()` — flush the 60-second TTL cache after writing the trust store.
- `TrustStoreError.code` — structured codes (`TRUST_STORE_FILE_MISSING`, `TRUST_STORE_INSECURE_PERMS`, `TRUST_STORE_DID_UNKNOWN`, `TRUST_STORE_BAD_KEY`, `TRUST_STORE_BAD_SCHEMA`, `TRUST_STORE_BAD_TOML`).
- `KEY_SIZE = 32`, `SIGNATURE_SIZE = 64` — never hard-code these numbers in your own code; import them.

**Operating principles:**
- One Ed25519 implementation in Arc. Always go through `arctrust.keypair`.
- `verify()` returns `False` on any failure — no try/except needed.
- Trust files live at `0o600`. Anything looser is rejected.
- Rotate by appending a new DID. Never reuse a DID with a different key.
- After any trust-file edit, call `invalidate_cache()` so live readers see it.

**API surface covered in this notebook:**
`KeyPair`, `KeyPair.from_seed`, `generate_keypair`, `sign`, `verify`,
`KEY_SIZE`, `SIGNATURE_SIZE`, `load_operator_pubkey`, `load_issuer_pubkey`,
`invalidate_cache`, `TrustStoreError`, `AgentIdentity`, `AgentIdentity.sign`,
`generate_did`.

**Next:** see `walkthroughs/arctrust/03-policy-pipeline.ipynb` for how these
signed primitives compose into the five-layer first-DENY-wins policy
evaluator that gates every tool call in Arc.